In [2]:
from pathlib import Path
import pandas as pd
from itables import show


In [3]:
from linkstates.visualise import plot_utilisation_by_departures, save_plot_html
from linkstates.analyse import collect_linkstate_utilisation_from_output_folder
from zugfolgezeit.gzb_theoretical_utilisation import compute_theoretical_utilisation_df
from delay_quantiles.gzb_delay_quantiles import compute_summary_runs_delay_quantiles


In [4]:
# ============================================================
# CONFIG – change these and hit ▶ Run
# ============================================================

filer = Path(r"/Users/nicolasdulex/devsbb/GZB_analysis")
#run_id = "output_20260217_it5_n200"
run_id = "output_20260219_it5_n5" #run with the new version that contains the statistics in the summary runs

OUTPUT_ROOT = filer / run_id

## Time window for the analysis (in seconds, usually the second hour of the simulation (3600-7200s))
WINDOW_START_S = 3600
WINDOW_END_S = 7200

PRINT_EVERY_S = 30

OUT_CSV = OUTPUT_ROOT / f"linkstate_utilisation_{run_id}.csv"

In [5]:
# ============================================================
# RUN OR LOAD CSV of LinkState utilisation (one row per link, per time window, per departure)
# ============================================================

if OUT_CSV.exists():
    print(f"[SKIP] CSV already exists → loading\n{OUT_CSV}")
    df_links = pd.read_csv(OUT_CSV)

else:
    print(f"[RUN] CSV not found → running linkstate analysis")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    df_links = collect_linkstate_utilisation_from_output_folder(
        OUTPUT_ROOT,
        window_start=WINDOW_START_S,
        window_end=WINDOW_END_S,
        print_every_s=PRINT_EVERY_S,
    )

    df_links.to_csv(OUT_CSV, index=False)
    print(f"[DONE] Wrote {len(df_links):,} rows to {OUT_CSV.resolve()}")

df_links

[RUN] CSV not found → running linkstate analysis
[INFO] Found 438 railsimLinkStates files under /Users/nicolasdulex/devsbb/GZB_analysis/output_20260219_it5_n5
[PROGRESS] 438/438 | last=0.01s | avg~=0.01s/file | elapsed=00:01 | ETA=00:00 | uc_2/uc2_bb2/04_simulation_run_output/transit_trunk/volume_12/uc2_bb2_transit_trunk_volume_12_sample_2/ITERS/it.0/uc2_bb2_transit_trunk_volume_12_sample_2.0.railsimLinkStates.csv.gz
[DONE] Wrote 6,962 rows to /Users/nicolasdulex/devsbb/GZB_analysis/output_20260219_it5_n5/linkstate_utilisation_output_20260219_it5_n5.csv


,use_case,building_block,operating_mode,volume,sample,link,exhausted_time_s,utilisation,utilisation_pct,departures,linkstate_file
0,uc_0,uc0_bb1,express_pass,volume_11,uc0_bb1_express_pass_volume_11_sample_1,f2,1682.0,0.467222,46.722222,22.0,uc_0/uc0_bb1/04_simulation_run_output/express_...
1,uc_0,uc0_bb1,express_pass,volume_11,uc0_bb1_express_pass_volume_11_sample_1,f3,1714.0,0.476111,47.611111,22.0,uc_0/uc0_bb1/04_simulation_run_output/express_...
2,uc_0,uc0_bb1,express_pass,volume_11,uc0_bb1_express_pass_volume_11_sample_1,f4,1512.0,0.420000,42.000000,22.0,uc_0/uc0_bb1/04_simulation_run_output/express_...
3,uc_0,uc0_bb1,express_pass,volume_11,uc0_bb1_express_pass_volume_11_sample_1,f5,1816.0,0.504444,50.444444,22.0,uc_0/uc0_bb1/04_simulation_run_output/express_...
4,uc_0,uc0_bb1,express_pass,volume_11,uc0_bb1_express_pass_volume_11_sample_1,f6,1984.0,0.551111,55.111111,22.0,uc_0/uc0_bb1/04_simulation_run_output/express_...
...,...,...,...,...,...,...,...,...,...,...,...
6957,uc_2,uc2_bb2,transit_trunk,volume_12,uc2_bb2_transit_trunk_volume_12_sample_2,r13,1872.0,0.520000,52.000000,24.0,uc_2/uc2_bb2/04_simulation_run_output/transit_...
6958,uc_2,uc2_bb2,transit_trunk,volume_12,uc2_bb2_transit_trunk_volume_12_sample_2,r14,1014.0,0.281667,28.166667,24.0,uc_2/uc2_bb2/04_simulation_run_output/transit_...
6959,uc_2,uc2_bb2,transit_trunk,volume_12,uc2_bb2_transit_trunk_volume_12_sample_2,r15,1872.0,0.520000,52.000000,24.0,uc_2/uc2_bb2/04_simulation_run_output/transit_...
6960,uc_2,uc2_bb2,transit_trunk,volume_12,uc2_bb2_transit_trunk_volume_12_sample_2,r16,1190.0,0.330556,33.055556,24.0,uc_2/uc2_bb2/04_simulation_run_output/transit_...


In [6]:
df_summary_runs = pd.read_csv(OUTPUT_ROOT / "output_run_summary.csv")

In [7]:
show(df_summary_runs)

Loading ITables v2.7.0 from the internet... (need help?)


In [8]:
# Links of interest per building block
BB_LINKS: dict[str, list[str]] = {
    "uc0_bb1": ["f4"],
    "uc0_bb2": ["f4"],
    "uc1_bb1": ["f05"],
    "uc1_bb2": ["f06", "f07"],
    "uc1_bb3": ["f10", "f11", "f12"],
    "uc2_bb1": ["f04", "r17"], #To discuss these are for now only the three switches (one link per switch but as they are linked with the ressource it should work)
    "uc2_bb2": ["f04", "r16"], #To discuss these are for now only the two switches (one link per switch but as they are linked with the ressource it should work) + the link to the ressource. I do not know how to identify which we need
    # fallback / catch-all is handled separately
}

In [9]:
# Filter the links of interest based on the building block and link IDs defined in BB_LINKS
df_links_of_interest = df_links.copy()

df_links_of_interest["use_case"] = df_links_of_interest["use_case"].astype(str)
df_links_of_interest["building_block"] = df_links_of_interest["building_block"].astype(str)
df_links_of_interest["link"] = df_links_of_interest["link"].astype(str)

def is_link_of_interest(row) -> bool:
    links = BB_LINKS.get(row["building_block"])
    if links is None:
        return False
    return row["link"] in links


df_filtered = df_links_of_interest[df_links_of_interest.apply(is_link_of_interest, axis=1)]
df_filtered.head()



,use_case,building_block,operating_mode,volume,sample,link,exhausted_time_s,utilisation,utilisation_pct,departures,linkstate_file
2,uc_0,uc0_bb1,express_pass,volume_11,uc0_bb1_express_pass_volume_11_sample_1,f4,1512.0,0.420000,42.000000,22.0,uc_0/uc0_bb1/04_simulation_run_output/express_...
7,uc_0,uc0_bb1,express_pass,volume_11,uc0_bb1_express_pass_volume_11_sample_2,f4,1510.0,0.419444,41.944444,22.0,uc_0/uc0_bb1/04_simulation_run_output/express_...
12,uc_0,uc0_bb1,express_pass,volume_12,uc0_bb1_express_pass_volume_12_sample_1,f4,1666.0,0.462778,46.277778,24.0,uc_0/uc0_bb1/04_simulation_run_output/express_...
17,uc_0,uc0_bb1,express_pass,volume_12,uc0_bb1_express_pass_volume_12_sample_2,f4,1570.0,0.436111,43.611111,24.0,uc_0/uc0_bb1/04_simulation_run_output/express_...
22,uc_0,uc0_bb1,express_pass,volume_13,uc0_bb1_express_pass_volume_13_sample_1,f4,1804.0,0.501111,50.111111,26.0,uc_0/uc0_bb1/04_simulation_run_output/express_...


we seem to have the same utilisation between the two methods
TODO : make a real analysis to see if we have something that is not the same

In [ ]:
df_merge = df_filtered.merge(df_summary_runs, left_on=["use_case", "building_block"], right_on=["use_case", "building_block"], how="left")